# LMS: CRUD Operations


In [1]:
from mysql.connector import Error

from db import friendly_error, run_command, run_query

## Students

In [2]:
student_id = run_command(
    "INSERT INTO students (full_name, email, phone, enrollment_date, status) VALUES (%s, %s, %s, %s, %s)",
    ("Sheikh Awais Rehman", "awais.rehman@example.com", "0300-7788991", "2025-01-20", "Active"),
)
student_id

21

In [3]:
run_query("SELECT * FROM students WHERE student_id = %s", (student_id,))

,student_id,full_name,email,phone,enrollment_date,status
0,21,Sheikh Awais Rehman,awais.rehman@example.com,0300-7788991,2025-01-20,Active


In [4]:
run_query("SELECT student_id, full_name, email FROM students WHERE full_name LIKE %s ORDER BY full_name", ("%Khan%",))

,student_id,full_name,email
0,10,Maryam Khan,maryam.khan@example.com


In [5]:
run_command("UPDATE students SET phone = %s, status = %s WHERE student_id = %s", ("0300-1122334", "Inactive", student_id))
run_query("SELECT full_name, phone, status FROM students WHERE student_id = %s", (student_id,))

,full_name,phone,status
0,Sheikh Awais Rehman,0300-1122334,Inactive


## Instructors

In [6]:
instructor_id = run_command(
    "INSERT INTO instructors (full_name, email, specialization, status) VALUES (%s, %s, %s, %s)",
    ("Ms. Hina Rashid", "hina.rashid@example.com", "Cyber Security", "Active"),
)
run_query("SELECT * FROM instructors WHERE instructor_id = %s", (instructor_id,))

,instructor_id,full_name,email,specialization,status
0,6,Ms. Hina Rashid,hina.rashid@example.com,Cyber Security,Active


In [7]:
run_command("UPDATE instructors SET specialization = %s WHERE instructor_id = %s", ("Network Security", instructor_id))
run_query("SELECT full_name, specialization FROM instructors WHERE instructor_id = %s", (instructor_id,))

,full_name,specialization
0,Ms. Hina Rashid,Network Security


## Courses

In [8]:
course_id = run_command(
    "INSERT INTO courses (course_name, description, instructor_id, duration_weeks, fee, status) VALUES (%s, %s, %s, %s, %s, %s)",
    ("Ethical Hacking Basics", "Introduction to network and web security testing", instructor_id, 8, 32000, "Active"),
)
run_query("""
    SELECT c.course_id, c.course_name, i.full_name AS instructor, c.duration_weeks, c.fee
    FROM courses c
    INNER JOIN instructors i ON i.instructor_id = c.instructor_id
    WHERE c.course_id = %s
""", (course_id,))

,course_id,course_name,instructor,duration_weeks,fee
0,8,Ethical Hacking Basics,Ms. Hina Rashid,8,32000.0


In [9]:
run_command("UPDATE courses SET fee = %s, duration_weeks = %s WHERE course_id = %s", (35000, 10, course_id))
run_query("SELECT course_name, fee, duration_weeks FROM courses WHERE course_id = %s", (course_id,))

,course_name,fee,duration_weeks
0,Ethical Hacking Basics,35000.0,10


## Enrollments

In [10]:
enrollment_id = run_command(
    "INSERT INTO enrollments (student_id, course_id, enrollment_date, status) VALUES (%s, %s, %s, %s)",
    (student_id, course_id, "2025-02-01", "Enrolled"),
)
run_query("""
    SELECT s.full_name AS student, c.course_name AS course, e.enrollment_date, e.status
    FROM enrollments e
    INNER JOIN students s ON s.student_id = e.student_id
    INNER JOIN courses c ON c.course_id = e.course_id
    WHERE e.enrollment_id = %s
""", (enrollment_id,))

,student,course,enrollment_date,status
0,Sheikh Awais Rehman,Ethical Hacking Basics,2025-02-01,Enrolled


In [11]:
run_command("UPDATE enrollments SET status = %s WHERE enrollment_id = %s", ("Completed", enrollment_id))
run_query("SELECT enrollment_id, status FROM enrollments WHERE enrollment_id = %s", (enrollment_id,))

,enrollment_id,status
0,38,Completed


## Grades

`total_marks` is calculated by MySQL from the three marks.

In [12]:
grade_id = run_command(
    "INSERT INTO grades (enrollment_id, assignment_marks, quiz_marks, final_marks) VALUES (%s, %s, %s, %s)",
    (enrollment_id, 16, 14, 47),
)
run_query("SELECT * FROM grades WHERE grade_id = %s", (grade_id,))

,grade_id,enrollment_id,assignment_marks,quiz_marks,final_marks,total_marks
0,33,38,16.0,14.0,47.0,77.0


In [13]:
run_command("UPDATE grades SET final_marks = %s WHERE grade_id = %s", (52, grade_id))
run_query("SELECT assignment_marks, quiz_marks, final_marks, total_marks FROM grades WHERE grade_id = %s", (grade_id,))

,assignment_marks,quiz_marks,final_marks,total_marks
0,16.0,14.0,52.0,82.0


## Validation and constraints

Each statement below should be rejected by the database.

In [14]:
tests = [
    ("Duplicate student email",
     "INSERT INTO students (full_name, email, phone, enrollment_date) VALUES (%s, %s, %s, %s)",
     ("Another Awais", "awais.rehman@example.com", "0300-0000000", "2025-01-21")),
    ("Duplicate enrollment",
     "INSERT INTO enrollments (student_id, course_id, enrollment_date) VALUES (%s, %s, %s)",
     (student_id, course_id, "2025-03-01")),
    ("Marks above the limit",
     "UPDATE grades SET assignment_marks = %s WHERE grade_id = %s",
     (25, grade_id)),
    ("Status outside the allowed list",
     "UPDATE students SET status = %s WHERE student_id = %s",
     ("Paused", student_id)),
    ("Course with a student that does not exist",
     "INSERT INTO enrollments (student_id, course_id, enrollment_date) VALUES (%s, %s, %s)",
     (9999, course_id, "2025-03-01")),
    ("Delete an instructor who teaches a course",
     "DELETE FROM instructors WHERE instructor_id = %s",
     (instructor_id,)),
]

for title, sql, params in tests:
    try:
        run_command(sql, params)
        print(f"{title}: accepted, check the table definition")
    except Error as error:
        print(f"{title}: {friendly_error(error)}")

Duplicate student email: This record already exists (duplicate email, name or enrollment).
Duplicate enrollment: This record already exists (duplicate email, name or enrollment).
Marks above the limit: One of the values is outside the allowed range.
Status outside the allowed list: Database error: Data truncated for column 'status' at row 1
Course with a student that does not exist: The selected related record does not exist.
Delete an instructor who teaches a course: This record cannot be deleted because other records depend on it.


## Delete the  records



In [15]:
print("Enrollments deleted:", run_command("DELETE FROM enrollments WHERE enrollment_id = %s", (enrollment_id,)))
print("Courses deleted:", run_command("DELETE FROM courses WHERE course_id = %s", (course_id,)))
print("Instructors deleted:", run_command("DELETE FROM instructors WHERE instructor_id = %s", (instructor_id,)))
print("Students deleted:", run_command("DELETE FROM students WHERE student_id = %s", (student_id,)))

Enrollments deleted: 1
Courses deleted: 1
Instructors deleted: 1
Students deleted: 1


In [16]:
run_query("SELECT COUNT(*) AS grades_left_for_deleted_enrollment FROM grades WHERE grade_id = %s", (grade_id,))

,grades_left_for_deleted_enrollment
0,0
